# Random Forest


O objetivo do projeto é prever a pontuação dos vinhos usando o algoritmo de Regressão de Random Forest para classificação multiclasse a partir de uma base de dados de avaliações de vinhos.
A base contém algumas características químicas do vinho que influemciam na sua qualidade.

**Caracaterísticas químicas do vinho:**
- Características dos Vinhos (Features)
- Fixed Acidity: Acidez fixa do vinho.
- Volatile Acidity: Acidez volátil do vinho.
- Citric Acid: Quantidade de ácido cítrico no vinho.
- Residual Sugar: Açúcar residual presente no vinho.
- Chlorides: Nível de cloretos no vinho.
- Free Sulfur Dioxide: Dióxido de enxofre livre no vinho.
- Total Sulfur Dioxide: Quantidade total de dióxido de enxofre no vinho.
- Density: Densidade do vinho.
- pH: Nível de pH do vinho.
- Sulphates: Quantidade de sulfatos no vinho.
- Alcohol: Teor alcoólico do vinho.

**Qualidade do Vinho (Variável de Saída,Target):**

Quality: Pontuação do vinho baseada em dados sensoriais, variando de 0 a 10.


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("winequality-red.csv", delimiter=',')

df.head(10)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5
6,7.9,0.60,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5
7,7.3,0.65,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7
8,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7
9,7.5,0.50,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.80,10.5,5


# Pré processamento dos dados.

## Verifcação da base 
Todas as variáveis estão no formato adequado, não há dados ausentes ou nulos

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [9]:
print(f"\nCampos com valores nulos:\n{df.isnull().sum()}")


Campos com valores nulos:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64


# 2 - Realize a segunda e terceita etapa de pré processamento dos dados.

A) Utilize a função describe para identificarmos outliers e verificarmos a distribuição dos dados.

B) Verifique o balanceamento da váriavel Target.

C)  Plote o gráfico ou a tabela e indique as variáveis que te parecem mais "fortes" na correlação para nosso modelo.

D) Crie um novo dataframe apenas com as váriaveis que parecem ter maior correlação com a target. (Negativa ou positiva)


## Verificação de Outliers
fixed acidity: valor máximo superior ao limite 14

In [11]:
df['fixed acidity'].describe()

count    1599.000000
mean        8.319637
std         1.741096
min         4.600000
25%         7.100000
50%         7.900000
75%         9.200000
max        15.900000
Name: fixed acidity, dtype: float64

In [12]:
df = df[df['fixed acidity'] <= 14]
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000,1591.000000
mean,8.284538,0.528143,0.269510,2.532715,0.087463,15.901948,46.546826,0.996723,3.312872,0.657555,10.421213,5.634821
std,1.673064,0.178907,0.193986,1.405646,0.047178,10.474248,32.937178,0.001858,0.152680,0.169678,1.059742,0.806745
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996720,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997800,3.400000,0.730000,11.100000,6.000000
max,14.000000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.000000,8.000000


mean + 2*std para agilizar a análise
a ideia é que 2var para além da média não deve ser muito distante de 75 nem max, se for, algo estranho está aocntecendo e vale avaliar mais
uma análise um pouco subjetiva, mas para uma avaliação rápida, pode ajudar


In [42]:
campos = ["75%", "Méd + 2*var", "Méd + 2*var", "Max"]
da = pd.DataFrame(columns=campos)
for campo in df:
    da.loc[campo] = [
        df[campo].describe()['75%'],
        df[campo].mean() + 2*df[campo].std(),
        df[campo].mean() + 3*df[campo].std(),
        df[campo].max()
    ]
da.head()

,75%,Méd + 2*var,Méd + 2*var,Max
fixed acidity,9.200,11.633473,13.308118,14.000
volatile acidity,0.635,0.873905,1.047847,1.185
citric acid,0.420,0.657711,0.851579,1.000
residual sugar,2.600,5.348380,6.755415,15.500
chlorides,0.090,0.181884,0.229104,0.611


as variáveis que me chamarm a atenção foram:
volatlie acidity
residual sugar
clorides
total sulfur dioxide
vamos olhar o histograma de cada um

In [34]:
import plotly.express as px


In [ ]:
px.histogram(df,
             x='volatile acidity',
             title='Histograma de Volatile Acidity',
             color_discrete_sequence=['indianred'],
             labels={'volatile acidity': 'Volatile Acidity'},
             barmode="group",
             histnorm="percent", 
             nbins=60).show()

In [ ]:
df = df[df['volatile acidity']<=1.2]
print("  75% /  M + 2*var / M + 2*var  /  Max")
for campo in df:
    print("\n{}\n{:.2f} / {:.2f} / {:.2f} / {:.2f} ".format(
        campo,
        df[campo].describe()['75%'],
        df[campo].mean() + 2*df[campo].std(),
        df[campo].mean() + 3*df[campo].std(),
        df[campo].max()
    ))

  75% /  M + 2*var / M + 2*var  /  Max

fixed acidity
9.20 / 11.63 / 13.31 / 14.00 

volatile acidity
0.64 / 0.87 / 1.05 / 1.19 

citric acid
0.42 / 0.66 / 0.85 / 1.00 

residual sugar
2.60 / 5.35 / 6.76 / 15.50 

chlorides
0.09 / 0.18 / 0.23 / 0.61 

free sulfur dioxide
21.00 / 36.85 / 47.31 / 72.00 

total sulfur dioxide
62.00 / 112.22 / 145.06 / 289.00 

density
1.00 / 1.00 / 1.00 / 1.00 

pH
3.40 / 3.62 / 3.77 / 4.01 

sulphates
0.73 / 1.00 / 1.17 / 2.00 

alcohol
11.10 / 12.54 / 13.60 / 14.00 

quality
6.00 / 7.25 / 8.05 / 8.00 


In [38]:
px.histogram(df,
             x='residual sugar',
             title='Histograma de Residual Sugar',
             color_discrete_sequence=['indianred'],
             labels={'residual sugar': 'Residual Sugar'},
             barmode="group",
             histnorm="percent", 
             nbins=60).show()

# 3 - Preparação Final dos Dados

A) Separe a base em X(Features) e Y(Target)

B) Separe a base em treino e teste.


In [ ]:
#seu código aqui

# 4 - Modelagem

A) Inicie e treine o modelo de Random Forest

B) Aplique a base de teste o modelo.


In [ ]:
#seu código aqui

# 5 - Avaliação

A) Avalie as principais métricas da Claissificação e traga insights acerca do resultado, interprete os valores achados.

B) Você nota que o modelo teve dificuldade para prever alguma classe? Se sim, acredita que tenha relação com o balanceamento dos dados? Explique.


In [ ]:
#seu código aqui

# 5 - Melhorando os Hyperparametros

A) Defina o Grid de parametros que você quer testar

B) Inicie e Treine um novo modelo utilizando o random search.

C) Avalie os resultados do modelo.

D) Você identificou melhorias no modelo após aplicar o random search? Justifique.


ps. Essa parte da atividade demorará um pouco para rodar!

In [ ]:
#seu código aqui

# 6 - Chegando a perfeição

Baseado em tudo que você já aprendeu até agora, quais outras técnicas você acredita que poderiam ser aplicadas ao modelo para melhorar ainda mais suas previsões?